In [25]:
import os
import cv2
import math
import random
import numpy as np
import albumentations as A
import shutil
from pathlib import Path

In [ ]:
INPUT_DIR = Path("dataset_cleaned/train_original")
OUTPUT_DIR = Path("dataset_cleaned/train")
TARGET_IMAGES = 600
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

In [27]:
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

In [ ]:
transform = A.Compose([
    A.Rotate(limit=15,p=0.7),
    A.RandomResizedCrop(size=(512,512),scale=(0.85,1.0),ratio=(0.9,1.1),p=0.4),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=(-0.1,0.15),contrast_limit=(-0.1,0.15)),
        A.RandomGamma(gamma_limit=(90,120))
    ],p=0.6),
    A.CLAHE(clip_limit=2.0,tile_grid_size=(16,16),p=0.2),
    A.GaussianBlur(blur_limit=(3,5),p=0.3),
    A.GaussNoise(std_range=(0.01,0.03),p=0.3)
],
seed=SEED
)

In [29]:
def get_image_paths(dir):
    return list(dir.rglob("*.png"))

In [30]:
def save_image(img,path):
    path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(path), img)

In [31]:
def augment(img_paths,output_path,imgs_needed):
    for img_path in img_paths:
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        save_image(img, output_path / img_path.name)
    for i in range(imgs_needed):
        img_path = img_paths[i%len(img_paths)]
        img = cv2.imread(str(img_path),cv2.IMREAD_GRAYSCALE)
        aug=transform(image=img)["image"]
        output_name=f"{img_path.stem}_aug{i:05d}.png"
        save_image(aug, output_path / output_name)
        

In [32]:
classes = list(d for d in INPUT_DIR.iterdir() if d.is_dir())

for c in classes:
    paths = get_image_paths(c)
    imgs_needed = max(0, TARGET_IMAGES - len(paths))
    output_path = OUTPUT_DIR / c.name
    print(f"creating {imgs_needed} images for {c.name} class")
    augment(paths,output_path,imgs_needed)

creating 490 images for Biliary_Leaks class
creating 95 images for Lithiasis class
creating 403 images for Normal class
creating 345 images for Stricture class
